In [ ]:
! python -V

In [ ]:
# OK
! pip install catboost

In [ ]:
# xAI
! pip install shap

In [ ]:
! pip install matplotlib

In [ ]:
# Import Prérequis
import os
import glob as glob
import numpy as np
import rasterio
from rasterio import plot
from rasterio.plot import show
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import BoundaryNorm, ListedColormap
#from matplotlib.cm import get_cmap
from matplotlib.axes import Axes
from IPython.display import HTML
from skimage import exposure
import rasterio as rio
from rasterio.plot import show
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sklearn
from sklearn.metrics import classification_report,accuracy_score

In [ ]:
REPERTOIRE_IMAGES = "D:\\PhD\\DATA\\FIGURES"
REPERTOIRE_DATASET = "F:\\DATA\\FOLDER"

# WORKAROUND POUR PROBLEME FICHIER UTILISÉ PAR PROCESSUS APRÈS RÉ-EXÉCUTION
# RENOMMER A CHAQUE EXECUTION (APRES AVOIR FAIT DES COPIES) AU LIEU DE TUER PROCESSUS OU DE REDEMARRER 
REPERTOIRE_POINTS = "POI_shp_north.bj6"
REPERTOIRE_SHAPE_10x10 = "Shp_10x10-Copie6"
REPERTOIRE_SHAPE_15x15 = "Shp_15x15-Copie6"
REPERTOIRE_SHAPE_20x20 = "Shp_20x20-Copie6"
REPERTOIRE_SHAPE_25x25 = "Shp_25x25-Copie6"
REPERTOIRE_SHAPE_30x30 = "Shp_30x30-Copie6"
REPERTOIRE_SHAPE_35x35 = "Shp_35x35-Copie6"
REPERTOIRE_SHAPE_40x40 = "Shp_40x40-Copie6"
REPERTOIRE_SHAPE_45x45 = "Shp_45x45-Copie6"
REPERTOIRE_SHAPE_50x50 = "Shp_50x50-Copie6"
REPERTOIRE_EXPORT_CSV = REPERTOIRE_DATASET + "\\" + "Export_CSV"

REPERTOIRE_SR = "SR" # valeur "SR". Répertoire destination SR : composite et VIs

ALGO_FUSION = 'dwt' # valeurs possibles "nsct", "dwt", "pwd" (suivant meilleur algorithme)

# ABLATION STUDY : si oui, (anciens) traitements existants, sinon += variables environnementales
HYPERPARAMS = True
ABLATION_STUDY = False

# TODO SAR
POLARISATION = "VH"

# NOM_SATELLITE
# valeurs possibles : "sentinel-2" ou "landsat-9" ou "planet" (composite)
# ou "S2_SR" (Super Resolution)
# ou "S2-L9" (Data Fusion simple (DF) ou Data Fusion Super Résolution (DF_SR) : différence dans le répertoire sat.)
# TODO SAR : projection de référence suivant satellite(s)
# ou "sentinel-1" (composite)
# TODO NOM_SATELLITE et TYPE_TRAITEMENT pour SAR
# ou "S1-S2" ou KO."S1-planet" ou KO."S1-SR" (sar-norm_S2-composite-norm)
# ou "S1-S2.VI" ou KO."S1-planet.VI" ou KO."S1-SR.VI" (sar-norm_S2-ndvi-ndwi)
# ou "S1-S2-S2.VI" ou KO."S1-planet-planet.VI" ou KO."S1-SR-SR.VI"
NOM_SATELLITE = "S1-S2-S2.VI"


# TYPE_TRAITEMENT
# valeurs possibles : 5 types
# simples: OK"composite" (S2, L9, Planet, SR, S1/SAR), OK"ndvi-ndwi" (S2, Planet, SR), KO"ndvi-ndwi_normalised"
# Optique mixte (composite treatment combined to vegetation indices) : OK"composite-norm_ndvi-ndwi" (S2, Planet, SR, S1/SAR)
# fusion: OK"fusion_originale" (S2-L9, FUSION_SR) et KO"fusion_normalised"
# SAR mixte : répertoire satellite SAR + lecture même obervation autre satellite
# TODO NOM_SATELLITE et TYPE_TRAITEMENT pour SAR
# SAR mixte 1: OK"sar-norm_S2-composite-norm"	(S1.norm+S2.norm, KO->S1+Planet.norm, KO->S1+SR.norm)
# SAR mixte 2: OK"sar-norm_S2-ndvi-ndwi" 	(S1.norm+S2.VI, KO->S1.norm+Planet.VI, KO->S1.norm+SR.VI)
# SAR mixte 3: OK"sar-norm_S2-composite-norm_S-2-ndvi-ndwi" (S1.norm+S2.norm+S2.VI, KO->S1.norm+Planet.norm+Planet.VI, KO->S1.norm+SR.norm+SR.VI)
TYPE_TRAITEMENT = "sar-norm_S2-composite-norm_S-2-ndvi-ndwi"

# TYPE_SR (en cas de super resolution)
TYPE_SR = "linear" # VALEURS POSSIBLES : "linear", "bicubic", "cnn", "gan", "transformers", "self-algo"


# REPERTOIRE_SATELLITE
# valeurs possibles
# "S-2_DECOUPE_FINAL.OK" ou "L-9_DECOUPE_FINAL.OK"
# ou "Planet_DECOUPE_FINAL.OK"
# ou "SR_blue" pour super-résolution (4 bandes : on indique une bande et on exploite les 3 autres pour compatibilité avec la logique du code existant)
# ou "FUSION_SIMPLE" (non SR), ou "FUSION_SR"
# or "S-1_DECOUPE_FINAL_LAST.OK" (tous les cas sentinel-1 ou sentinel-1 mixte)
REPERTOIRE_SATELLITE = "S-1_DECOUPE_FINAL_LAST.OK" # pour conserver la compatibilité avec le code existant
REPERTOIRE_COMPLET_SATELLITE = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE

# Spécifités SR (cas non SAR)
REPERTOIRE_COMPLET_SR = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SR
REPERTOIRE_COMPLET_SATELLITE_BLUE = REPERTOIRE_COMPLET_SATELLITE
REPERTOIRE_COMPLET_SATELLITE_GREEN = REPERTOIRE_DATASET + "\\" + "SR_green"
REPERTOIRE_COMPLET_SATELLITE_RED = REPERTOIRE_DATASET + "\\" + "SR_red"
REPERTOIRE_COMPLET_SATELLITE_NIR = REPERTOIRE_DATASET + "\\" + "SR_nir"

# KO car résolution S1 != {Planet,SR} Répertoires mixtes SAR
REPERTOIRE_SATELLITE2 = "S-2_DECOUPE_FINAL.OK" # pour NOM_SATELLITE = "S1-S2" or "S1-S2.VI" or "S1-S2-S2.VI"
#KO car résolution S1 != Planet REPERTOIRE_SATELLITE2 = "Planet_DECOUPE_FINAL.OK" # pour NOM_SATELLITE = "S1-planet" or "S1-planet.VI" or "S1-planet-planet.VI"
#KO car résolution S1 != SR REPERTOIRE_SATELLITE2 = "SR_blue" # pour NOM_SATELLITE = "S1-SR" or "S1-SR.VI" or "S1-SR-SR.VI"
REPERTOIRE_COMPLET_SATELLITE2 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE2
# KO car résolution S1 != SR Spécificités SR (cas non combiné à SAR) // utilisation pour la lecture des valeurs de pixels et non pour la prjection (identique pour tous les SR)
REPERTOIRE_COMPLET_SATELLITE_BLUE2 = REPERTOIRE_COMPLET_SATELLITE2
REPERTOIRE_COMPLET_SATELLITE_GREEN2 = REPERTOIRE_DATASET + "\\" + "SR_green"
REPERTOIRE_COMPLET_SATELLITE_RED2 = REPERTOIRE_DATASET + "\\" + "SR_red"
REPERTOIRE_COMPLET_SATELLITE_NIR2 = REPERTOIRE_DATASET + "\\" + "SR_nir"


# EXTENSIONS SATELLITE
EXTENSION_SENTINEL2 = ".jp2"
EXTENSION_LANDSAT9 = ".TIF"
EXTENSION_PLANET = ".tif"
EXTENSION_FUSION = ".tif"
EXTENSION_SR = ".tif"
EXTENSION_SENTINEL1 = ".img"

REPERTOIRE_COMPLET_POINTS = REPERTOIRE_DATASET + "\\" + REPERTOIRE_POINTS

REPERTOIRE_COMPLET_SHAPE_10x10 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_10x10
REPERTOIRE_COMPLET_SHAPE_15x15 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_15x15
REPERTOIRE_COMPLET_SHAPE_20x20 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_20x20
REPERTOIRE_COMPLET_SHAPE_25x25 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_25x25
REPERTOIRE_COMPLET_SHAPE_30x30 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_30x30
REPERTOIRE_COMPLET_SHAPE_35x35 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_35x35
REPERTOIRE_COMPLET_SHAPE_40x40 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_40x40
REPERTOIRE_COMPLET_SHAPE_45x45 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_45x45
REPERTOIRE_COMPLET_SHAPE_50x50 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SHAPE_50x50

# Affichages
print("Nom satellite", NOM_SATELLITE)
print("Traitement", TYPE_TRAITEMENT)
print("Répertoire complet satellite", REPERTOIRE_COMPLET_SATELLITE)
print('HYPERPARAMETERS', HYPERPARAMS)
print("ABLATION STUDY", ABLATION_STUDY)
if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
    print("Algo de fusion", ALGO_FUSION)
if REPERTOIRE_SATELLITE == 'FUSION_SR' or (NOM_SATELLITE == "S2_SR") or (NOM_SATELLITE == "S1-SR") or (NOM_SATELLITE == "S1-SR.VI") or (NOM_SATELLITE == "S1-SR-SR.VI"):
    print("TYPE_SR", TYPE_SR)
    print("REPERTOIRE DESTINATION SR", REPERTOIRE_COMPLET_SR) # répertoire composite et VIs
    print("REPERTOIRE COMPLET SATELLITE BLUE", REPERTOIRE_COMPLET_SATELLITE_BLUE)
    print("REPERTOIRE COMPLET SATELLITE GREEN", REPERTOIRE_COMPLET_SATELLITE_GREEN)
    print("REPERTOIRE COMPLET SATELLITE RED", REPERTOIRE_COMPLET_SATELLITE_RED)
    print("REPERTOIRE COMPLET SATELLITE NIR", REPERTOIRE_COMPLET_SATELLITE_NIR)
print("Répertoires complets shapes")
print(REPERTOIRE_COMPLET_SHAPE_10x10)
print(REPERTOIRE_COMPLET_SHAPE_15x15)
print(REPERTOIRE_COMPLET_SHAPE_20x20)
print(REPERTOIRE_COMPLET_SHAPE_25x25)
print(REPERTOIRE_COMPLET_SHAPE_30x30)
print(REPERTOIRE_COMPLET_SHAPE_35x35)
print(REPERTOIRE_COMPLET_SHAPE_40x40)
print(REPERTOIRE_COMPLET_SHAPE_45x45)
print(REPERTOIRE_COMPLET_SHAPE_50x50)
print("Répertoire complet points", REPERTOIRE_COMPLET_POINTS)

In [ ]:
# LECTURE DES LABELS DES OBSERVATIONS
import pandas as pd
#df = pd.read_csv('D:\\PhD\\DATA\\Echantillon_data_IRC_attaque_jassides_coton_stats_NORD_partiel.csv', delimiter=';')
df = pd.read_csv('Severity_levels+Meteorological_data.csv', delimiter=';')
#print(df)

# FIELD_ID
dict_IRC_FieldID = df.to_dict(orient='index')
dict_IRC_FieldID_final = {}
length_fieldID = len(dict_IRC_FieldID)
for i in range(length_fieldID):
    id_dict = dict_IRC_FieldID[i]['ID']
    field_id = dict_IRC_FieldID[i]['ID_Parcelle']
    dict_IRC_FieldID_final[id_dict] = field_id
print('ID Parcelle', len(dict_IRC_FieldID_final), 'observations')
print(dict_IRC_FieldID_final)

# LABEL
dict_IRC_label = df.to_dict(orient='index')
dict_IRC_label_final = {}
length_label = len(dict_IRC_label)
for i in range(length_label):
    id_dict = dict_IRC_label[i]['ID']
    label_dict = dict_IRC_label[i]['Label']
    dict_IRC_label_final[id_dict] = label_dict
# POUR COMPATIBILITE AVEC ANCIEN CODE
dict_IRC_final = dict_IRC_label_final
print()
print('Label', len(dict_IRC_label_final), 'observations')
print(dict_IRC_label_final)

# Intervalle plus ou moins en jours pour LST
dict_IRC_intervalle_LST = df.to_dict(orient='index')
dict_IRC_intervalle_LST_final = {}
length_intervalle_LST = len(dict_IRC_intervalle_LST)
for i in range(length_intervalle_LST):
    id_dict = dict_IRC_intervalle_LST[i]['ID']
    intervalle_LST_dict = dict_IRC_intervalle_LST[i]['Intervalle_plus-moins_LST']
    dict_IRC_intervalle_LST_final[id_dict] = intervalle_LST_dict
print()
print('Intervalle LST', len(dict_IRC_intervalle_LST_final), 'observations')
print(dict_IRC_intervalle_LST_final)

# Distance en km pour LST
dict_IRC_dist_LST = df.to_dict(orient='index')
dict_IRC_dist_LST_final = {}
length_dist_LST = len(dict_IRC_dist_LST)
for i in range(length_dist_LST):
    id_dict = dict_IRC_dist_LST[i]['ID']
    dist_LST_dict = dict_IRC_dist_LST[i]['Dist_LST']
    dict_IRC_dist_LST_final[id_dict] = dist_LST_dict
print()
print('Distance LST', len(dict_IRC_dist_LST_final), 'observations')
print(dict_IRC_dist_LST_final)

# LST
dict_IRC_LST = df.to_dict(orient='index')
dict_IRC_LST_final = {}
length_LST = len(dict_IRC_LST)
for i in range(length_LST):
    id_dict = dict_IRC_LST[i]['ID']
    LST_dict = dict_IRC_LST[i]['LST']
    dict_IRC_LST_final[id_dict] = LST_dict
print()
print('LST', len(dict_IRC_dist_LST_final), 'observations')
print(dict_IRC_LST_final)

# Intervalle plus ou moins en jours pour Précipitation
dict_IRC_intervalle_Prec = df.to_dict(orient='index')
dict_IRC_intervalle_Prec_final = {}
length_intervalle_Prec = len(dict_IRC_intervalle_Prec)
for i in range(length_intervalle_Prec):
    id_dict = dict_IRC_intervalle_Prec[i]['ID']
    intervalle_Prec_dict = dict_IRC_intervalle_Prec[i]['Intervalle_plus-moins_Prec']
    dict_IRC_intervalle_Prec_final[id_dict] = intervalle_Prec_dict
print()
print('Intervalle Précipitation', len(dict_IRC_intervalle_Prec_final), 'observations')
print(dict_IRC_intervalle_Prec_final)

# Distance en km pour Précipitation
dict_IRC_dist_Prec = df.to_dict(orient='index')
dict_IRC_dist_Prec_final = {}
length_dist_Prec = len(dict_IRC_dist_Prec)
for i in range(length_dist_Prec):
    id_dict = dict_IRC_dist_Prec[i]['ID']
    dist_Prec_dict = dict_IRC_dist_Prec[i]['Dist_Prec']
    dict_IRC_dist_Prec_final[id_dict] = dist_Prec_dict
print()
print('Distance Précipitation', len(dict_IRC_dist_Prec_final), 'observations')
print(dict_IRC_dist_Prec_final)

# LST
dict_IRC_Precipitation = df.to_dict(orient='index')
dict_IRC_Precipitation_final = {}
length_Precipitation = len(dict_IRC_Precipitation)
for i in range(length_Precipitation):
    id_dict = dict_IRC_Precipitation[i]['ID']
    Precipitation_dict = dict_IRC_Precipitation[i]['Precipitation']
    dict_IRC_Precipitation_final[id_dict] = Precipitation_dict
print()
print('Précipitation', len(dict_IRC_dist_LST_final), 'observations')
print(dict_IRC_Precipitation_final)

In [ ]:
# CHOIX D'UNE PROJECTION

# POINT CENTRAL SHAPE 3 ha
points_dentrainements_original = REPERTOIRE_COMPLET_POINTS + "\\" + "Points_observation.shp"
points_dentrainements_reproj = REPERTOIRE_COMPLET_POINTS + "\\" + "Points_observation_reproj.shp"

# ouverture de l'image
# choix arbitraire image (elles ont toutes les mêmes projections)
if NOM_SATELLITE == "sentinel-2":
    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "T31PDM_20230707T100559_B02.jp2"
elif NOM_SATELLITE == "landsat-9":
    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "LC08_L1TP_192053_20230714_20230724_02_T1_B11.TIF"
elif NOM_SATELLITE == "planet":
    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "20230711_100105_39_2470_3B_AnalyticMS_SR_clip.tif"
elif NOM_SATELLITE == "S2-L9": # comme S2
    if REPERTOIRE_SATELLITE == "FUSION_SIMPLE":
        raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "46_fused_dwt.tif"
    elif REPERTOIRE_SATELLITE == "FUSION_SR":
        raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "46_fused_pwd_bicubic.tif"
    else:
        print("Mauvais choix de satellite S2-L9")
# SAR
elif NOM_SATELLITE == "sentinel-1":
    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "Gamma0_VH.img"
elif NOM_SATELLITE == "S1-S2" or NOM_SATELLITE == "S1-S2.VI" or NOM_SATELLITE == "S1-S2-S2.VI":
    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "Gamma0_VH.img"
    raster_loc2 = REPERTOIRE_COMPLET_SATELLITE2 + "\\46\\" + "T31PDM_20230707T100559_B02.jp2"
#elif NOM_SATELLITE == "S1-planet" or NOM_SATELLITE == "S1-planet.VI" or NOM_SATELLITE == "S1-planet-planet.VI":
#    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "Gamma0_VH.img"
#    raster_loc2 = REPERTOIRE_COMPLET_SATELLITE2 + "\\46\\" + "20230711_100105_39_2470_3B_AnalyticMS_SR_clip.tif"
#elif NOM_SATELLITE == "S1-SR" or NOM_SATELLITE == "S1-SR.VI" or NOM_SATELLITE == "S1-SR-SR.VI":
#    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\46\\" + "Gamma0_VH.img"
#    raster_loc2 = REPERTOIRE_COMPLET_SATELLITE2 + "\\46\\" + "46_sentinel2_sr_bicubic.tif"
# SUPER RESOLUTION (non combinée à SAR)
elif NOM_SATELLITE == "S2_SR": # comme Planet
    raster_loc = REPERTOIRE_COMPLET_SATELLITE + "\\" + "46" + "\\" + "46_sentinel2_sr_bicubic.tif"
else:
    print("Mauvais choix de satellite")

# AFFICHAGE DES VALEURS POUR VERIFICATION
if NOM_SATELLITE == "S1-S2" or NOM_SATELLITE == "S1-S2.VI" or NOM_SATELLITE == "S1-S2-S2.VI" \
    or NOM_SATELLITE == "S1-planet" or NOM_SATELLITE == "S1-planet.VI" or NOM_SATELLITE == "S1-planet-planet.VI" \
    or NOM_SATELLITE == "S1-SR" or NOM_SATELLITE == "S1-SR.VI" or NOM_SATELLITE == "S1-SR-SR.VI": \
    print("Chemin images satellite pour choix projection", raster_loc, raster_loc2)
else:
    print("Chemin images satellite pour choix projection", raster_loc)

In [ ]:
# PROJECTION : EXECUTER UNE FOIS POUR CHAQUE SATELLITE
stack_src = rasterio.open(raster_loc)
# projection du shapefile en concordance avec l'image satellite
shp1 = gpd.read_file(points_dentrainements_original)
profil_crs_img = stack_src.profile
shp_utm31N = shp1.to_crs(profil_crs_img["crs"])
# export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
shp_utm31N.to_file(points_dentrainements_reproj, mode="w")
stack_src.close()

In [ ]:
# fonction de normalisation pour la mise à l'échelle des valeurs des pixels entre 0 et 1
def normalise(array):
    array_min, array_max = array.min(), array.max()
    return((array - array_min)/(array_max - array_min))

In [ ]:
# /!\ BIEN VÉRIFIER QUE LE NOMBRE D'OBSERVATIONS CORRESPOND ET L'AFFICHAGE DES VALEURS DE PIXELS À LA FIN

# utilisation de df.append, bien que déconseillée, car il y a peu de lignes
data = pd.DataFrame()

listes_id = os.listdir(REPERTOIRE_COMPLET_SATELLITE)

# Initialisations
num_ligne_sampled = 0
sampled = pd.DataFrame()

nobs = 0
for id_rep in listes_id:
    print("Observation n° " + str(nobs+1) + " en cours : ID = " + str(id_rep))
    nobs = nobs + 1
    #Ouverture des bandes
    if NOM_SATELLITE == "sentinel-1":
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_SENTINEL1:
                nom_fichier_decoupe = nom_fichier.split("_")
                polarisation_img = nom_fichier_decoupe[-1]
                if polarisation_img == POLARISATION:
                    band = rasterio.open(repertoire_image + "\\" + image)

        #Traitement proprement dit
        #if TYPE_TRAITEMENT == "composite":
        # Metadonnées
        meta = band.meta
        meta.update({"count": 1})
        rgb_or_vi_path = repertoire_image + "\\" + id_rep + "_composite" + '.TIF'

        # supprimer le fichier pour contourner problème écrasement fichier existant
        if os.path.exists(rgb_or_vi_path):
            os.remove(rgb_or_vi_path)

        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
            dest.write(band.read(1),1)
    elif NOM_SATELLITE == "S1-S2": # "sar-norm_S2-composite-norm"
        # Sentinel
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_SENTINEL1:
                nom_fichier_decoupe = nom_fichier.split("_")
                polarisation_img = nom_fichier_decoupe[-1]
                if polarisation_img == POLARISATION:
                    band = rasterio.open(repertoire_image + "\\" + image)


        # Sentinel-2
        # liste des images du repertoire
        repertoire_image2 = REPERTOIRE_COMPLET_SATELLITE2 + "\\" + str(id_rep)
        liste_images_sat2 = os.listdir(repertoire_image2)

        # lecture des bandes suivant les fichiers correspondants
        for image2 in liste_images_sat2:
            split2 = os.path.splitext(image2)
            nom_fichier2 = split2[0]
            extension2 = split2[1]
            if extension2 == EXTENSION_SENTINEL2:
                nom_fichier_decoupe2 = nom_fichier2.split("_")
                bande2 = nom_fichier_decoupe2[-1]
                if bande2 == "B02":
                    band2 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B03":
                    band3 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B04":
                    band4 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B08":
                    band8 = rasterio.open(repertoire_image2 + "\\" + image2)

        # composite
        sar = band.read()
        blue = band2.read()
        green = band3.read()
        red = band4.read()
        nir = band8.read()
        # normalisation composite
        sarn = normalise(sar)
        bluen = normalise(blue)
        greenn = normalise(green)
        redn = normalise(red)
        nirn = normalise(nir)

        # Construction de l'image
        # Metadonnées
        meta = band3.meta
        meta.update({"count": 5})
        meta.update(driver='GTiff')
        meta.update(dtype=rasterio.float32)

        rgb_or_vi_path = repertoire_image + "\\" + id_rep + "_sar-norm_S2-composite-norm" + '.TIF'

        # supprimer le fichier pour contourner problème écrasement fichier existant
        if os.path.exists(rgb_or_vi_path):
            os.remove(rgb_or_vi_path)

        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
            dest.write(sarn.astype(rasterio.float32)[0], 1)
            dest.write(bluen.astype(rasterio.float32)[0], 2)
            dest.write(greenn.astype(rasterio.float32)[0], 3)
            dest.write(redn.astype(rasterio.float32)[0], 4)
            dest.write(nirn.astype(rasterio.float32)[0], 5)
    elif NOM_SATELLITE == "S1-S2.VI": # "sar-norm_S2-ndvi-ndwi"
        # Sentinel
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_SENTINEL1:
                nom_fichier_decoupe = nom_fichier.split("_")
                polarisation_img = nom_fichier_decoupe[-1]
                if polarisation_img == POLARISATION:
                    band = rasterio.open(repertoire_image + "\\" + image)


        # Sentinel-2
        # liste des images du repertoire
        repertoire_image2 = REPERTOIRE_COMPLET_SATELLITE2 + "\\" + str(id_rep)
        liste_images_sat2 = os.listdir(repertoire_image2)

        # lecture des bandes suivant les fichiers correspondants
        for image2 in liste_images_sat2:
            split2 = os.path.splitext(image2)
            nom_fichier2 = split2[0]
            extension2 = split2[1]
            if extension2 == EXTENSION_SENTINEL2:
                nom_fichier_decoupe2 = nom_fichier2.split("_")
                bande2 = nom_fichier_decoupe2[-1]
                if bande2 == "B02":
                    band2 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B03":
                    band3 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B04":
                    band4 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B08":
                    band8 = rasterio.open(repertoire_image2 + "\\" + image2)

        # composite
        sar = band.read()
        blue = band2.read()
        green = band3.read()
        red = band4.read()
        nir = band8.read()
        # normalisation composite
        sarn = normalise(sar)
        # NDVI : (nir - red)/(nir + red)
        ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)
        # NDWI : (green - nir)/(green + nir)
        ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)


        # Construction de l'image
        # Metadonnées
        meta = band3.meta
        meta.update({"count": 3})
        meta.update(driver='GTiff')
        meta.update(dtype=rasterio.float32)

        rgb_or_vi_path = repertoire_image + "\\" + id_rep + "_sar-norm_S2-ndvi-ndwi" + '.TIF'

        # supprimer le fichier pour contourner problème écrasement fichier existant
        if os.path.exists(rgb_or_vi_path):
            os.remove(rgb_or_vi_path)

        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
            dest.write(sarn.astype(rasterio.float32)[0], 1)
            dest.write(ndvi.astype(rasterio.float32)[0], 2)
            dest.write(ndwi.astype(rasterio.float32)[0], 3)
    elif NOM_SATELLITE == "S1-S2-S2.VI": # "sar-norm_S2-composite-norm_S-2-ndvi-ndwi"
        # Sentinel
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_SENTINEL1:
                nom_fichier_decoupe = nom_fichier.split("_")
                polarisation_img = nom_fichier_decoupe[-1]
                if polarisation_img == POLARISATION:
                    band = rasterio.open(repertoire_image + "\\" + image)


        # Sentinel-2
        # liste des images du repertoire
        repertoire_image2 = REPERTOIRE_COMPLET_SATELLITE2 + "\\" + str(id_rep)
        liste_images_sat2 = os.listdir(repertoire_image2)

        # lecture des bandes suivant les fichiers correspondants
        for image2 in liste_images_sat2:
            split2 = os.path.splitext(image2)
            nom_fichier2 = split2[0]
            extension2 = split2[1]
            if extension2 == EXTENSION_SENTINEL2:
                nom_fichier_decoupe2 = nom_fichier2.split("_")
                bande2 = nom_fichier_decoupe2[-1]
                if bande2 == "B02":
                    band2 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B03":
                    band3 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B04":
                    band4 = rasterio.open(repertoire_image2 + "\\" + image2)
                if bande2 == "B08":
                    band8 = rasterio.open(repertoire_image2 + "\\" + image2)

        # composite
        sar = band.read()
        blue = band2.read()
        green = band3.read()
        red = band4.read()
        nir = band8.read()
        # normalisation composite
        sarn = normalise(sar)
        bluen = normalise(blue)
        greenn = normalise(green)
        redn = normalise(red)
        nirn = normalise(nir)
        # NDVI : (nir - red)/(nir + red)
        ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)
        # NDWI : (green - nir)/(green + nir)
        ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)


        # Construction de l'image
        # Metadonnées
        meta = band3.meta
        meta.update({"count": 7})
        meta.update(driver='GTiff')
        meta.update(dtype=rasterio.float32)

        rgb_or_vi_path = repertoire_image + "\\" + id_rep + "_sar-norm_S2-composite-norm_S-2-ndvi-ndwi" + '.TIF'

        # supprimer le fichier pour contourner problème écrasement fichier existant
        if os.path.exists(rgb_or_vi_path):
            os.remove(rgb_or_vi_path)

        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
            dest.write(sarn.astype(rasterio.float32)[0], 1)
            dest.write(bluen.astype(rasterio.float32)[0], 2)
            dest.write(greenn.astype(rasterio.float32)[0], 3)
            dest.write(redn.astype(rasterio.float32)[0], 4)
            dest.write(nirn.astype(rasterio.float32)[0], 5)
            dest.write(ndvi.astype(rasterio.float32)[0], 6)
            dest.write(ndwi.astype(rasterio.float32)[0], 7)
    elif NOM_SATELLITE == "sentinel-2":
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_SENTINEL2:
                nom_fichier_decoupe = nom_fichier.split("_")
                bande = nom_fichier_decoupe[-1]
                if bande == "B02":
                    band2 = rasterio.open(repertoire_image + "\\" + image)
                if bande == "B03":
                    band3 = rasterio.open(repertoire_image + "\\" + image)
                if bande == "B04":
                    band4 = rasterio.open(repertoire_image + "\\" + image)
                if bande == "B08":
                    band8 = rasterio.open(repertoire_image + "\\" + image)

        #Traitement proprement dit
        if TYPE_TRAITEMENT == "composite":
            # Metadonnées
            meta = band2.meta
            meta.update({"count": 4})
            rgb_or_vi_path = repertoire_image + "\\" + id_rep + "_composite" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(band2.read(1),1)
                dest.write(band3.read(1),2)
                dest.write(band4.read(1),3)
                dest.write(band8.read(1),4)
        elif TYPE_TRAITEMENT == "ndvi-ndwi":
            # NDVI : (nir - red)/(nir + red)
            red = band4.read()
            nir = band8.read()
            ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)

            # NDWI : (green - nir)/(green + nir)
            green = band3.read()
            nir = band8.read()
            ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)

            # Construction de l'image NDVI-NDWI
            # Metadonnées
            meta = band3.meta
            meta.update({"count": 2})
            meta.update(driver='GTiff')
            meta.update(dtype=rasterio.float32)
            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_ndvi-ndwi" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(ndvi.astype(rasterio.float32)[0], 1)
                dest.write(ndwi.astype(rasterio.float32)[0], 2)
        elif TYPE_TRAITEMENT == "composite-norm_ndvi-ndwi":
            # composite
            blue = band2.read()
            green = band3.read()
            red = band4.read()
            nir = band8.read()
            # normalisation composite
            bluen = normalise(blue)
            greenn = normalise(green)
            redn = normalise(red)
            nirn = normalise(nir)

            # NDVI : (nir - red)/(nir + red)
            ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)

            # NDWI : (green - nir)/(green + nir)
            ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)

            # Construction de l'image NDVI-NDWI
            # Metadonnées
            meta = band3.meta
            meta.update({"count": 6})
            meta.update(driver='GTiff')
            meta.update(dtype=rasterio.float32)

            rgb_or_vi_path = repertoire_image + "\\" + id_rep + "_composite-norm_ndvi-ndwi" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(bluen.astype(rasterio.float32)[0], 1)
                dest.write(greenn.astype(rasterio.float32)[0], 2)
                dest.write(redn.astype(rasterio.float32)[0], 3)
                dest.write(nirn.astype(rasterio.float32)[0], 4)
                dest.write(ndvi.astype(rasterio.float32)[0], 5)
                dest.write(ndwi.astype(rasterio.float32)[0], 6)
        elif TYPE_TRAITEMENT == "ndvi-ndwi_normalised": # KO
            # NDVI_N : (nirn - redn)/(nirn + redn)
            red = band4.read()
            nir = band8.read()

            redn = normalise(red)
            nirn = normalise(nir)

            ndvin = (nirn.astype(float) - redn.astype(float)) / (nirn+redn)

            # NDWI : (green - nir)/(green + nir)
            green = band3.read()
            nir = band8.read()

            greenn = normalise(green)
            nirn = normalise(nir)

            ndwin = (greenn.astype(float) - nirn.astype(float)) / (greenn+nirn)

            # Construction de l'image NDVI-NDWI
            # Metadonnées
            meta = band3.meta
            meta.update({"count": 2})
            meta.update(driver='GTiff')
            meta.update(dtype=rasterio.float32)
            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_ndvi-ndwi_normalised" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(ndvin.astype(rasterio.float32)[0], 1)
                dest.write(ndwin.astype(rasterio.float32)[0], 2)
        else:
            print("Mauvais choix de type de traitement")

    elif NOM_SATELLITE == "landsat-9":
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_LANDSAT9:
                nom_fichier_decoupe = nom_fichier.split("_")
                bande = nom_fichier_decoupe[-1]
                if bande == "B10":
                    band10 = rasterio.open(repertoire_image + "\\" + image)
                if bande == "B11":
                    band11 = rasterio.open(repertoire_image + "\\" + image)


        if TYPE_TRAITEMENT == "composite":
            #Fusion proprement dite : Fichier composite
            # Metadonnées
            meta = band10.meta
            meta.update({"count": 2})
            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_composite" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(band10.read(1),1)
                dest.write(band11.read(1),2)
        else:
            print("Mauvais choix de type de traitement")
    elif NOM_SATELLITE == "planet":
        # liste des images du repertoire
        #repertoire_image = REPERTOIRE_COMPLET_SATELLITE + '\\' + str(id_rep) + '\\PSScene'
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + '\\' + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]

            if extension == EXTENSION_PLANET:
                fichier_img = nom_fichier.split("_")
                caracteristique_img = fichier_img[-2]
                if caracteristique_img == 'SR':
                    #lecture du fichier image
                    chemin_image = repertoire_image + "\\" + image
                    #src = rio.open(chemin_image)
                    if TYPE_TRAITEMENT == "composite":
                        rgb_or_vi_path = chemin_image
                    elif TYPE_TRAITEMENT == "ndvi-ndwi":
                        img = rasterio.open(chemin_image)

                        # lecture des bandes
                        blue = img.read(1)
                        green = img.read(2)
                        red = img.read(3)
                        nir = img.read(4)

                        # calcul de l'indice
                        ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)

                        # NDWI : (green - nir)/(green + nir)
                        ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)

                        # Construction de l'image NDWI
                        # Metadonnées
                        meta = img.meta
                        meta.update({"count": 2})
                        meta.update(driver='GTiff')
                        meta.update(dtype=rasterio.float32)
                        rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_ndvi-ndwi" + '.TIF'

                        # supprimer le fichier pour contourner problème écrasement fichier existant
                        if os.path.exists(rgb_or_vi_path):
                            os.remove(rgb_or_vi_path)

                        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                            dest.write(ndvi.astype(rasterio.float32), 1)
                            dest.write(ndwi.astype(rasterio.float32), 2)
                    elif TYPE_TRAITEMENT == "composite-norm_ndvi-ndwi":
                        # lecture des bandes
                        img = rasterio.open(chemin_image)

                        blue = img.read(1)
                        green = img.read(2)
                        red = img.read(3)
                        nir = img.read(4)

                        bluen = normalise(blue)
                        greenn = normalise(green)
                        redn = normalise(red)
                        nirn = normalise(nir)

                        # calcul de l'indice
                        ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)
                        # NDWI : (green - nir)/(green + nir)
                        ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)

                        # Construction de l'image NDWI
                        # Metadonnées
                        meta = img.meta
                        meta.update({"count": 6})
                        meta.update(driver='GTiff')
                        meta.update(dtype=rasterio.float32)
                        rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_composite-norm_ndvi-ndwi" + '.TIF'

                        # supprimer le fichier pour contourner problème écrasement fichier existant
                        if os.path.exists(rgb_or_vi_path):
                            os.remove(rgb_or_vi_path)

                        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                            dest.write(bluen.astype(rasterio.float32), 1)
                            dest.write(greenn.astype(rasterio.float32), 2)
                            dest.write(redn.astype(rasterio.float32), 3)
                            dest.write(nirn.astype(rasterio.float32), 4)
                            dest.write(ndvi.astype(rasterio.float32), 5)
                            dest.write(ndwi.astype(rasterio.float32), 6)
                    elif TYPE_TRAITEMENT == "ndvi-ndwi_normalised":
                        # lecture des bandes
                        img = rasterio.open(chemin_image)

                        blue = img.read(1)
                        green = img.read(2)
                        red = img.read(3)
                        nir = img.read(4)

                        redn = normalise(red)
                        nirn = normalise(nir)
                        greenn = normalise(green)

                        # calcul de l'indice
                        ndvin = (nirn.astype(float) - redn.astype(float)) / (nirn+redn)

                        # calcul de l'indice
                        # NDWIn : (greenn - nirn)/(greenn + nirn)
                        ndwin = (greenn.astype(float) - nirn.astype(float)) / (greenn+nirn)

                        # Construction de l'image NDWI
                        # Metadonnées
                        meta = img.meta
                        meta.update({"count": 2})
                        meta.update(driver='GTiff')
                        meta.update(dtype=rasterio.float32)
                        rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_ndvi-ndwi_normalised" + '.TIF'

                        # supprimer le fichier pour contourner problème écrasement fichier existant
                        if os.path.exists(rgb_or_vi_path):
                            os.remove(rgb_or_vi_path)

                        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                            dest.write(ndvin.astype(rasterio.float32), 1)
                            dest.write(ndwin.astype(rasterio.float32), 2)
                    else:
                        print("Mauvais choix de type de traitement")

    elif NOM_SATELLITE == "S2_SR":
        # liste des images du repertoire (SR_blue, SR_green, SR_red, SR_nir)
        repertoire_image_blue = REPERTOIRE_COMPLET_SATELLITE_BLUE + "\\" + str(id_rep) # REPERTOIRE_COMPLET_SATELLITE_BLUE = REPERTOIRE_COMPLET_SATELLITE
        repertoire_image_green = REPERTOIRE_COMPLET_SATELLITE_GREEN + "\\" + str(id_rep)
        repertoire_image_red = REPERTOIRE_COMPLET_SATELLITE_RED + "\\" + str(id_rep)
        repertoire_image_nir = REPERTOIRE_COMPLET_SATELLITE_NIR + "\\" + str(id_rep)
        # FAIRE LISTING D'UN REPERTOIRE (1 bande SR) ET UTILISER UN CHEMIN EQUIVALENT POUR LES AUTRES BANDES
        liste_images_sat = os.listdir(repertoire_image_blue)

        # Répertoire de destination SR
        repertoire_image = REPERTOIRE_COMPLET_SR + "\\" + str(id_rep)
        if not os.path.exists(repertoire_image):
            os.mkdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_SR:
                nom_fichier_decoupe = nom_fichier.split("_")
                type_sr = nom_fichier_decoupe[-1]
                if type_sr == TYPE_SR:
                    band2 = rasterio.open(repertoire_image_blue + "\\" + image)
                    band3 = rasterio.open(repertoire_image_green + "\\" + image)
                    band4 = rasterio.open(repertoire_image_red + "\\" + image)
                    band8 = rasterio.open(repertoire_image_nir + "\\" + image)

        #Traitement proprement dit
        if TYPE_TRAITEMENT == "composite":
            # Metadonnées
            meta = band2.meta
            meta.update({"count": 4})
            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_" + TYPE_SR + "_composite" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(band2.read(1),1)
                dest.write(band3.read(1),2)
                dest.write(band4.read(1),3)
                dest.write(band8.read(1),4)
        elif TYPE_TRAITEMENT == "ndvi-ndwi":
            # NDVI : (nir - red)/(nir + red)
            red = band4.read()
            nir = band8.read()
            ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)

            # NDWI : (green - nir)/(green + nir)
            green = band3.read()
            nir = band8.read()
            ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)

            # Construction de l'image NDVI-NDWI
            # Metadonnées
            meta = band3.meta
            meta.update({"count": 2})
            meta.update(driver='GTiff')
            meta.update(dtype=rasterio.float32)
            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_" + TYPE_SR + "_ndvi-ndwi" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(ndvi.astype(rasterio.float32)[0], 1)
                dest.write(ndwi.astype(rasterio.float32)[0], 2)
        elif TYPE_TRAITEMENT == "composite-norm_ndvi-ndwi":
            # composite
            blue = band2.read()
            green = band3.read()
            red = band4.read()
            nir = band8.read()
            # normalisation composite
            bluen = normalise(blue)
            greenn = normalise(green)
            redn = normalise(red)
            nirn = normalise(nir)

            # NDVI : (nir - red)/(nir + red)
            ndvi = (nir.astype(float) - red.astype(float)) / (nir+red)

            # NDWI : (green - nir)/(green + nir)
            ndwi = (green.astype(float) - nir.astype(float)) / (green+nir)

            # Construction de l'image NDVI-NDWI
            # Metadonnées
            meta = band3.meta
            meta.update({"count": 6})
            meta.update(driver='GTiff')
            meta.update(dtype=rasterio.float32)
            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_" + TYPE_SR + "_composite-norm_ndvi-ndwi" + '.TIF'

            # supprimer le fichier pour contourner problème écrasement fichier existant
            if os.path.exists(rgb_or_vi_path):
                os.remove(rgb_or_vi_path)

            with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                dest.write(bluen.astype(rasterio.float32)[0], 1)
                dest.write(greenn.astype(rasterio.float32)[0], 2)
                dest.write(redn.astype(rasterio.float32)[0], 3)
                dest.write(nirn.astype(rasterio.float32)[0], 4)
                dest.write(ndvi.astype(rasterio.float32)[0], 5)
                dest.write(ndwi.astype(rasterio.float32)[0], 6)

    elif NOM_SATELLITE == "S2-L9":
        # liste des images du repertoire
        repertoire_image = REPERTOIRE_COMPLET_SATELLITE + "\\" + str(id_rep)
        liste_images_sat = os.listdir(repertoire_image)

        # lecture des bandes suivant les fichiers correspondants
        for image in liste_images_sat:
            split = os.path.splitext(image)
            nom_fichier = split[0]
            extension = split[1]
            if extension == EXTENSION_FUSION:
                chemin_img = repertoire_image + "\\" + image
                if TYPE_TRAITEMENT == "fusion_originale":
                    nom_fichier_decoupe = nom_fichier.split("_")
                    if REPERTOIRE_SATELLITE == "FUSION_SIMPLE":
                        algo_fusion_fichier = nom_fichier_decoupe[-1]
                        if ALGO_FUSION == algo_fusion_fichier:
                            rgb_or_vi_path = chemin_img # quel que soit le type de traitement
                    elif REPERTOIRE_SATELLITE == "FUSION_SR":
                        type_sr = nom_fichier_decoupe[-1]
                        algo_fusion_fichier = nom_fichier_decoupe[-2]
                        if ALGO_FUSION == algo_fusion_fichier and TYPE_SR == type_sr:
                            rgb_or_vi_path = chemin_img # quel que soit le type de traitement
                elif TYPE_TRAITEMENT == "fusion_normalised":
                    nom_fichier_decoupe = nom_fichier.split("_")
                    if REPERTOIRE_SATELLITE == "FUSION_SIMPLE":
                        algo_fusion_fichier = nom_fichier_decoupe[-1]
                    elif REPERTOIRE_SATELLITE == "FUSION_SR":
                        type_sr = nom_fichier_decoupe[-1]
                        algo_fusion_fichier = nom_fichier_decoupe[-2]
                    else:
                        print("MAUVAIS CHOIX DE METHODE DE FUSION")
                    if ALGO_FUSION == algo_fusion_fichier:
                        img_fusion = rasterio.open(chemin_img)
                        bande_fusion = img_fusion.read()
                        bande_fusion_norm = normalise(bande_fusion)

                        # Construction de l'image fusionnée normalisée
                        # Metadonnées
                        meta = img_fusion.meta
                        meta.update(driver='GTiff')
                        meta.update(dtype=rasterio.float32)
                        if REPERTOIRE_SATELLITE == "FUSION_SIMPLE":
                            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_fusion_normalised" + EXTENSION_FUSION
                        elif REPERTOIRE_SATELLITE == "FUSION_SR":
                            rgb_or_vi_path = repertoire_image + "\\" + str(id_rep) + "_fusion_normalised_" + TYPE_SR + EXTENSION_FUSION
                        else:
                            print("MAUVAIS CHOIX DE METHODE DE FUSION")

                        # supprimer le fichier pour contourner problème écrasement fichier existant
                        if os.path.exists(rgb_or_vi_path):
                            os.remove(rgb_or_vi_path)

                        with rasterio.open(rgb_or_vi_path, 'w', **meta) as dest:
                            dest.write(bande_fusion_norm.astype(rasterio.float32)[0], 1)

                        img_fusion.close()
                else:
                    print("Mauvais choix de type de traitement")
    else:
       print("Nom de satellite incorrect")



    if NOM_SATELLITE == 'planet':
        nom_dossier_decoupe = id_rep.split("_")
        id = nom_dossier_decoupe[0]
    else:
        id = id_rep

    # Shape de tailles différentes
    shape_dentrainements_original_50x50 = REPERTOIRE_COMPLET_SHAPE_50x50 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_50x50 = REPERTOIRE_COMPLET_SHAPE_50x50 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_45x45 = REPERTOIRE_COMPLET_SHAPE_45x45 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_45x45 = REPERTOIRE_COMPLET_SHAPE_45x45 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_40x40 = REPERTOIRE_COMPLET_SHAPE_40x40 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_40x40 = REPERTOIRE_COMPLET_SHAPE_40x40 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_35x35 = REPERTOIRE_COMPLET_SHAPE_35x35 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_35x35 = REPERTOIRE_COMPLET_SHAPE_35x35 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_30x30 = REPERTOIRE_COMPLET_SHAPE_30x30 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_30x30 = REPERTOIRE_COMPLET_SHAPE_30x30 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_25x25 = REPERTOIRE_COMPLET_SHAPE_25x25 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_25x25 = REPERTOIRE_COMPLET_SHAPE_25x25 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_20x20 = REPERTOIRE_COMPLET_SHAPE_20x20 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_20x20 = REPERTOIRE_COMPLET_SHAPE_20x20 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_15x15 = REPERTOIRE_COMPLET_SHAPE_15x15 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_15x15 = REPERTOIRE_COMPLET_SHAPE_15x15 + "\\" + str(id) + "_reproj23.shp" # test de reprojection
    shape_dentrainements_original_10x10 = REPERTOIRE_COMPLET_SHAPE_10x10 + "\\" + str(id) + ".shp" # shape carré 1/4 ha
    shape_dentrainements_reproj_10x10 = REPERTOIRE_COMPLET_SHAPE_10x10 + "\\" + str(id) + "_reproj23.shp" # test de reprojection

    # points_dentrainements_original = REPERTOIRE_COMPLET_POINTS + "\\" + 'pt_' + str(id) + ".shp" # shape carré 1/4 ha
    # points_dentrainements_reproj = REPERTOIRE_COMPLET_POINTS + "\\" + 'pt-_' + str(id) + "_reproj23.shp" # test de reprojection


    # ouverture de l'image
    raster_loc = rgb_or_vi_path
    stack_src = rasterio.open(raster_loc)
    profil_crs_img = stack_src.profile
    #print(profil_crs_img)

    # projection du shapefile du carré en concordance avec l'image satellite
    # 50x50
    shp1_50x50 = gpd.read_file(shape_dentrainements_original_50x50)
    shp_utm31N_50x50 = shp1_50x50.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_50x50.to_file(shape_dentrainements_reproj_50x50, mode="w")
    # 45x45
    shp1_45x45 = gpd.read_file(shape_dentrainements_original_45x45)
    shp_utm31N_45x45 = shp1_45x45.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_45x45.to_file(shape_dentrainements_reproj_45x45, mode="w")
    # 40x40
    shp1_40x40 = gpd.read_file(shape_dentrainements_original_40x40)
    shp_utm31N_40x40 = shp1_40x40.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_40x40.to_file(shape_dentrainements_reproj_40x40, mode="w")
    # 35x35
    shp1_35x35 = gpd.read_file(shape_dentrainements_original_35x35)
    shp_utm31N_35x35 = shp1_35x35.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_35x35.to_file(shape_dentrainements_reproj_35x35, mode="w")
    # 30x30
    shp1_30x30 = gpd.read_file(shape_dentrainements_original_30x30)
    shp_utm31N_30x30 = shp1_30x30.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_30x30.to_file(shape_dentrainements_reproj_30x30, mode="w")
    # 25x25
    shp1_25x25 = gpd.read_file(shape_dentrainements_original_25x25)
    shp_utm31N_25x25 = shp1_25x25.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_25x25.to_file(shape_dentrainements_reproj_25x25, mode="w")
    # 20x20
    shp1_20x20 = gpd.read_file(shape_dentrainements_original_20x20)
    shp_utm31N_20x20 = shp1_20x20.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_20x20.to_file(shape_dentrainements_reproj_20x20, mode="w")
    # 15x15
    shp1_15x15 = gpd.read_file(shape_dentrainements_original_15x15)
    shp_utm31N_15x15 = shp1_15x15.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_15x15.to_file(shape_dentrainements_reproj_15x15, mode="w")
    # 10x10
    shp1_10x10 = gpd.read_file(shape_dentrainements_original_10x10)
    shp_utm31N_10x10 = shp1_10x10.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    shp_utm31N_10x10.to_file(shape_dentrainements_reproj_10x10, mode="w")

    # Déjà fait dans cellule plus haut
    # projection du shapefile des points en concordance avec l'image satellite
    #shp_pt = gpd.read_file(points_dentrainements_original)
    #profil_crs_img = stack_src.profile
    #shp_utm31N2 = shp_pt.to_crs(profil_crs_img["crs"])
    # export du shp reprojeté (on écrase pour être sûr que le shp corresponde bien aux caractéristiques de l'image satellite)
    #shp_utm31N2.to_file(points_dentrainements_reproj, mode="w")

    # ouverture du shp reprojé avec fiona
    shp2_10x10 = fiona.open(shape_dentrainements_reproj_10x10)
    shp2_15x15 = fiona.open(shape_dentrainements_reproj_15x15)
    shp2_20x20 = fiona.open(shape_dentrainements_reproj_20x20)
    shp2_25x25 = fiona.open(shape_dentrainements_reproj_25x25)
    shp2_30x30 = fiona.open(shape_dentrainements_reproj_30x30)
    shp2_35x35 = fiona.open(shape_dentrainements_reproj_35x35)
    shp2_40x40 = fiona.open(shape_dentrainements_reproj_40x40)
    shp2_45x45 = fiona.open(shape_dentrainements_reproj_45x45)
    shp2_50x50 = fiona.open(shape_dentrainements_reproj_50x50)

    # lecture des réflectances et IVs


    # Initialisations
    num_ligne_tmp = 0
    siteID = id



    # DATA AUGMENTATION
    # AND ENVIRONMENTAL VARIABLES LST & Precipitation
    # 2 square layers for S-2 (20 * 20 et 30*30), 1 square layer for L9 (50*50)

    # 50x50 : Landsat 9, Planet, SR, SR+DF
    if NOM_SATELLITE == "landsat-9" or NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" \
      or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        for feature in shp2_50x50:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 45x45 : Planet, SR, SR+DF
        for feature in shp2_45x45:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)
                
                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
    # 40x40 : Planet, SR, SR+DF
        for feature in shp2_40x40:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 35x35 : Planet, SR, SR+DF
        for feature in shp2_35x35:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "sentinel-1" or NOM_SATELLITE == "S1-S2" or NOM_SATELLITE == "S1-S2.VI" or NOM_SATELLITE == "S1-S2-S2.VI" \
      or NOM_SATELLITE == "sentinel-2" or NOM_SATELLITE == "planet" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SIMPLE") or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 30x30 : Sentinel-1, Sentinel-2, Planet, S2-L9 (DF), SR, SR+DF
        for feature in shp2_30x30:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 25x25 : Planet, SR, SR+DF
        for feature in shp2_25x25:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "sentinel-1" or NOM_SATELLITE == "S1-S2" or NOM_SATELLITE == "S1-S2.VI" or NOM_SATELLITE == "S1-S2-S2.VI" \
      or NOM_SATELLITE == "sentinel-2" or NOM_SATELLITE == "planet" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SIMPLE") or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 20x20 : Sentinel-1, Sentinel-2, Planet, S2-L9 (DF), SR, SR+DF
        for feature in shp2_20x20:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 15x15 : Planet, SR, SR+DF
        for feature in shp2_15x15:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1

    if NOM_SATELLITE == "planet" or NOM_SATELLITE == "S2_SR" or (NOM_SATELLITE == "S2-L9" and REPERTOIRE_SATELLITE == "FUSION_SR"):
        # 10x10 : Planet, SR, SR+DF
        for feature in shp2_10x10:
            # siteID = feature['properties']['ID']
            #label = feature['properties']['Label']
            label = dict_IRC_final[int(id)]
            lst = dict_IRC_LST_final[int(id)]
            precipitation = dict_IRC_Precipitation_final[int(id)]
            ### Field_ID
            field_id = dict_IRC_FieldID_final[int(id)]
            coords = feature['geometry']['coordinates']

            num_coord = 1
            nbr_coord = len(coords[0])
            for coord in coords[0]:
                ###
                sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
                ligne = []
                ligne.append(siteID)
                ligne.append(label)
                ligne.append(lst)
                ligne.append(precipitation)
                ###
                ligne.append(field_id)

                # bandes
                value = [v for v in stack_src.sample([coord])]
                nbr_bandes = len(value[0])
                for i in range(nbr_bandes):
                    ligne.append(value[0][i])
                    # ajout colonne pour chaque bande
                    sampled_tmp['bande' + str(i+1)] = pd.Series(dtype='float')
                if num_coord != nbr_coord:
                    # valeurs dans le df_tmp
                    sampled_tmp.loc[num_ligne_tmp + num_coord] = ligne
                    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

                num_coord = num_coord + 1

            num_ligne_tmp = num_ligne_tmp + 1





    # Point central
    point = fiona.open(points_dentrainements_reproj)
    ###
    sampled_tmp = pd.DataFrame(columns = ['ID', 'Label', 'LST', 'Precipitation', 'ID_Parcelle'])
    ligne = []
    ligne.append(siteID)
    ligne.append(label)
    ligne.append(lst)
    ligne.append(precipitation)
    ###
    ligne.append(field_id)

    for feature2 in point:
        if feature2['properties']['ID'] == int(id):
            coord2 = feature2['geometry']['coordinates']
            value2 = [v for v in stack_src.sample(coord2)]
    nbr_bandes2 = len(value2[0])
    for j in range(nbr_bandes2):
        ligne.append(value2[0][j])
        # ajout colonne pour chaque bande
        sampled_tmp['bande' + str(j+1)] = pd.Series(dtype='float')
    sampled_tmp.loc[num_ligne_tmp + num_coord + 1] = ligne
    sampled = pd.concat([sampled, sampled_tmp], ignore_index = True)

    # fermeture du fichier image
    stack_src.close()

print(nobs, 'observations traitées')

sampled

In [ ]:
severities = ['Severity 1','Severity 2','Severity 3','Severity 4']

In [ ]:
# 1)
# Raccourci Pour ne pas reprendre l'exécution depuis le début pour un même traitement
ABLATION_STUDY = True

In [ ]:
# 2)
# Raccourci Pour ne pas reprendre l'exécution depuis le début pour un même traitement
ABLATION_STUDY = False

In [ ]:
# renommer les bandes selon le type de satellite et le traitement
# et aussi mettre les bandes selon le bon ordre
df = pd.DataFrame()


### /!\ LABEL DERNIERE COLONNE POUR ETRE (facilement) CONSIDEREE COMME y

if NOM_SATELLITE == "sentinel-1": # "composite" SAR
    df['ID_Parcelle'] = sampled['ID_Parcelle']
    df['ID'] = sampled['ID']
    df['SAR'] = sampled['bande1'] #
    feature_names = ['SAR']
    if ABLATION_STUDY == False:
        df['LST'] = sampled['LST']
        df['Precipitation'] = sampled['Precipitation']
        feature_names.append('LST')
        feature_names.append('Precipitation')
    df['Label'] = sampled['Label']
elif NOM_SATELLITE == "S1-S2": # sar-norm_S2-composite-norm
    df['ID_Parcelle'] = sampled['ID_Parcelle']
    df['ID'] = sampled['ID']
    df['SARn'] = sampled['bande1']
    df['bluen'] = sampled['bande2'] # band 2 S-2
    df['greenn'] = sampled['bande3'] # band 3 S-2
    df['redn'] = sampled['bande4'] # band 4 S-2
    df['nirn'] = sampled['bande5'] # band 8 S-2
    feature_names = ['SAR normalized', 'Blue normalized', 'Green normalized', 'Red normalized', 'NIR normalized']
    if ABLATION_STUDY == False:
        df['LST'] = sampled['LST']
        df['Precipitation'] = sampled['Precipitation']
        feature_names.append('LST')
        feature_names.append('Precipitation')
    df['Label'] = sampled['Label']
elif NOM_SATELLITE == "S1-S2.VI": # "sar-norm_S2-ndvi-ndwi
    df['ID_Parcelle'] = sampled['ID_Parcelle']
    df['ID'] = sampled['ID']
    df['SARn'] = sampled['bande1']
    df['ndvi'] = sampled['bande2']
    df['ndwi'] = sampled['bande3']
    feature_names = ['SAR normalized', 'NDVI', 'NDWI']
    if ABLATION_STUDY == False:
        df['LST'] = sampled['LST']
        df['Precipitation'] = sampled['Precipitation']
        feature_names.append('LST')
        feature_names.append('Precipitation')
    df['Label'] = sampled['Label']
elif NOM_SATELLITE == "S1-S2-S2.VI": # sar-norm_S2-composite-norm_S-2-ndvi-ndwi
    df['ID_Parcelle'] = sampled['ID_Parcelle']
    df['ID'] = sampled['ID']
    df['SARn'] = sampled['bande1']
    df['bluen'] = sampled['bande2'] # band 2 S-2
    df['greenn'] = sampled['bande3'] # band 3 S-2
    df['redn'] = sampled['bande4'] # band 4 S-2
    df['nirn'] = sampled['bande5'] # band 8 S-2
    df['ndvi'] = sampled['bande6']
    df['ndwi'] = sampled['bande7']
    feature_names = ['SAR normalized', 'Blue normalized', 'Green normalized', 'Red normalized', 'NIR normalized', 'NDVI', 'NDWI']
    if ABLATION_STUDY == False:
        df['LST'] = sampled['LST']
        df['Precipitation'] = sampled['Precipitation']
        feature_names.append('LST')
        feature_names.append('Precipitation')
    df['Label'] = sampled['Label']
elif NOM_SATELLITE == "sentinel-2":
    if TYPE_TRAITEMENT == "composite":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['blue'] = sampled['bande1'] # band 2 S-2
        df['green'] = sampled['bande2'] # band 3 S-2
        df['red'] = sampled['bande3'] # band 4 S-2
        df['nir'] = sampled['bande4'] # band 8 S-2
        feature_names = ['Blue', 'Green', 'Red', 'NIR']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "ndvi-ndwi":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['ndvi'] = sampled['bande1']
        df['ndwi'] = sampled['bande2']
        feature_names = ['NDVI', 'NDWI']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "composite-norm_ndvi-ndwi":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['bluen'] = sampled['bande1'] # band 2 S-2
        df['greenn'] = sampled['bande2'] # band 3 S-2
        df['redn'] = sampled['bande3'] # band 4 S-2
        df['nirn'] = sampled['bande4'] # band 8 S-2
        df['ndvi'] = sampled['bande5']
        df['ndwi'] = sampled['bande6']
        feature_names = ['Blue normalized', 'Green normalized', 'Red normalized', 'NIR normalized', 'NDVI', 'NDWI']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    # KO
    elif TYPE_TRAITEMENT == "ndvi-ndwi_normalised":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['ndvi_normalised'] = sampled['bande1']
        df['ndwi_normalised'] = sampled['bande2']
        df['Label'] = sampled['Label']
    else:
        print("Mauvais choix de traitement S-2")
elif NOM_SATELLITE == "planet":
    if TYPE_TRAITEMENT == "composite":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['blue'] = sampled['bande1'] # band 2 S-2
        df['green'] = sampled['bande2'] # band 3 S-2
        df['red'] = sampled['bande3'] # band 4 S-2
        df['nir'] = sampled['bande4'] # band 8 S-2
        feature_names = ['Blue', 'Green', 'Red', 'NIR']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "ndvi-ndwi":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['ndvi'] = sampled['bande1']
        df['ndwi'] = sampled['bande2']
        feature_names = ['NDVI', 'NDWI']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "composite-norm_ndvi-ndwi":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['bluen'] = sampled['bande1'] # band 2 S-2
        df['greenn'] = sampled['bande2'] # band 3 S-2
        df['redn'] = sampled['bande3'] # band 4 S-2
        df['nirn'] = sampled['bande4'] # band 8 S-2
        df['ndvi'] = sampled['bande5']
        df['ndwi'] = sampled['bande6']
        feature_names = ['Blue normalized', 'Green normalized', 'Red normalized', 'NIR normalized', 'NDVI', 'NDWI']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    # KO
    elif TYPE_TRAITEMENT == "ndvi-ndwi_normalised":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['ndvi_normalised'] = sampled['bande1']
        df['ndwi_normalised'] = sampled['bande2']
        df['Label'] = sampled['Label']
    else:
        print("Mauvais choix de traitement Planet")
elif NOM_SATELLITE == "landsat-9":
    if TYPE_TRAITEMENT == "composite":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['TIRS1'] = sampled['bande1'] # band 10 landsat-9
        df['TIRS2'] = sampled['bande2'] # band 11 landsat-9
        feature_names = ['TIRS1', 'TIRS2']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    else:
        print("Mauvais choix de traitement Landsat-9")
elif NOM_SATELLITE == "S2_SR":
    if TYPE_TRAITEMENT == "composite":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['blue'] = sampled['bande1'] # band 2 S-2
        df['green'] = sampled['bande2'] # band 3 S-2
        df['red'] = sampled['bande3'] # band 4 S-2
        df['nir'] = sampled['bande4'] # band 8 S-2
        feature_names = ['Blue', 'Green', 'Red', 'NIR']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "ndvi-ndwi":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['ndvi'] = sampled['bande1']
        df['ndwi'] = sampled['bande2']
        feature_names = ['NDVI', 'NDWI']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "composite-norm_ndvi-ndwi":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['bluen'] = sampled['bande1'] # band 2 S-2
        df['greenn'] = sampled['bande2'] # band 3 S-2
        df['redn'] = sampled['bande3'] # band 4 S-2
        df['nirn'] = sampled['bande4'] # band 8 S-2
        df['ndvi'] = sampled['bande5']
        df['ndwi'] = sampled['bande6']
        feature_names = ['Blue normalized', 'Green normalized', 'Red normalized', 'NIR normalized', 'NDVI', 'NDWI']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
elif NOM_SATELLITE == "S2-L9":
    if TYPE_TRAITEMENT == "fusion_originale":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['Fused_Original'] = sampled['bande1'] # band fusionnée S2 + L9
        feature_names = ['Fused Simple']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    elif TYPE_TRAITEMENT == "fusion_normalised":
        df['ID_Parcelle'] = sampled['ID_Parcelle']
        df['ID'] = sampled['ID']
        df['Fused_Normalised'] = sampled['bande1'] # band fusionnée S2 + L9
        feature_names = ['Fused Normalized']
        if ABLATION_STUDY == False:
            df['LST'] = sampled['LST']
            df['Precipitation'] = sampled['Precipitation']
            feature_names.append('LST')
            feature_names.append('Precipitation')
        df['Label'] = sampled['Label']
    else:
        print("Mauvais choix de traitement S2-L9")
else:
    print("Mauvais choix de satellite")
    
print('Feature names', feature_names)
df

In [ ]:
# Suppression de la colonne ID (pour ne pas entraîner l'algo avec l'ID)
df2 = df.drop(columns = ['ID'])
df2

In [ ]:
# CHOIX DE BANDES
# 1. WITHOUT FIELD_ID : CAS A) (ABLATION_STUDY==TRUE uniquement), CAS B)
#########################(le cas A, apporté à titre de comparaison, ne prenait pas en compte les variables environnementales)
# 2. WITH FIELD_ID : CAS C)
# 3. MOST CORRELATED BANDS

# LEGENDE
# CAS A)
##### OK LAST SANS ID_PARCELLE SANS KFOLD

# CAS B)
##### OK LAST SANS ID_PARCELLE AVEC KFOLD

# CAS C)
##### OK LAST ak ID_PARCELLE + KFOLD

In [ ]:
# 1. TOUTES BANDES NON ID_PARCELLE
x=df2.iloc[:,1:df2.shape[1] - 1] # sans Parcelle ID
X=x.values
y=df2.iloc[:,-1]
Y= y.values
print(X.shape)
print(Y.shape)

In [ ]:
# 2. TOUTES BANDES AVEC ID_PARCELLE
x=df2.iloc[:,1:df2.shape[1] - 1]
y=df2.iloc[:,-1]

# IDs PARCELLE
groups = (df2.iloc[:,0]).values
print(x.shape)
print(y.shape)

In [ ]:
# 3. CHOIX BANDE CORRELEE
num_bande = 2 # valeurs possibles : 1..4 pour S-2 et 1..2 pour L9
x=df2.iloc[:, num_bande - 1] # 0 bande1, ...
X=x.values
y=df2.iloc[:,-1]
Y= y.values
print(X.shape)
print(Y.shape)

In [ ]:
# CAS A)
##### OK LAST SANS ID_PARCELLE SANS KFOLD
############### //!!!\\ METTRE ABLATION_STUDY = True pour ne pas prendre en compte variables environnementales
########### ///!!!!!!\\\\\\\\\ EXECUTER # TOUTES BANDES NON PARCELLE_ID
# TOUTES BANDES TEST SPLIT ak KFold + SHAP +/- ABLATION_STUDY
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score


# TEST SPLIT
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,Y,test_size=0.20,stratify=Y) # 80/20 <-> 5-Fold
#print(f'X_train Shape:{X_train.shape}\nX_test Shape:{X_test.shape}\ny_train Shape:{y_train.shape}\ny_test Shape:{y_test.shape}')


# /!\ FAIT SEULEMENT POUR RF
algo = 'RF' # les valeurs sont 'SVM', 'RF' et 'CatBoost' ou implémenter le choix d'autres algorithmes au besoin

# TRAITEMENT GLOBAL
print('NO FIELD ID WITHOUT STRATIFIED K-FOLD - TOUTES BANDES TEST SPLIT SANS VALIDATION')
print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
if NOM_SATELLITE == 'S2_SR':
    print("TYPE_SR", TYPE_SR)
if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
    print("Algo de fusion", ALGO_FUSION)
    if REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("TYPE_SR", TYPE_SR)
if ABLATION_STUDY == True: # PAS D'ABLATION STUDY DANS NOTRE CAS
    print('ABLATION STUDY')

if algo == 'SVM':
    # SVM
    from sklearn.svm import SVC

    # choix de l'algo
    cName=algo
    clf= SVC(kernel='rbf', class_weight='balanced', probability=True)

    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)

    # Overall Acuracy,  Kappa ; Precision; rappel, F1 score
    print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
    if NOM_SATELLITE == 'S2_SR':
        print("TYPE_SR", TYPE_SR)
    if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("Algo de fusion", ALGO_FUSION)
        if REPERTOIRE_SATELLITE == 'FUSION_SR':
            print("TYPE_SR", TYPE_SR)
    print(f"Précision globale {cName} : {accuracy_score(y_test,clf_pred)*100}")
    print(f"Indice de Kappa {cName} : {cohen_kappa_score(y_test,clf_pred)*100}")
    print(classification_report(y_test,clf_pred))

elif algo == 'RF':
    # Random Forest
    from sklearn.ensemble import RandomForestClassifier
    
    from sklearn.model_selection import train_test_split

    # choix de l'algo
    cName=algo

    #i = 1

    # from scratch model to avoid data leakage
    # manual hyperparams
    #param_grid = {
    #        'n_estimators': [50, 100, 150],  # Number of trees
    #        'max_depth': [None, 10, 20],     # Maximum depth of trees
    #        'min_samples_split': [2, 5],     # Minimum samples required to split a node
    #        'min_samples_leaf': [1, 2],      # Minimum samples required to be at a leaf node
    #        'max_features': ['auto', 'sqrt', 'log2']  # Features to consider for best split
    #    }

    clf= RandomForestClassifier(
        class_weight='balanced',# balanced : penalization algorithm
        #n_estimators=100,
        #max_depth=20,
        #min_samples_split=5,
        #min_samples_leaf=1,
        #random_state=50, # reproductibilité, mais biais car modèle non iniatialisé à vide à chaque essai
    )
    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)
    # f1-score
    score = f1_score(y_test, clf_pred, average='weighted')
    print('f1-score:', score * 100, '%')
    print(classification_report(y_test,clf_pred))
    
elif algo == 'CatBoost':
    # XGBoost
    from catboost import CatBoostClassifier
    from sklearn.utils.class_weight import compute_class_weight

    # choix de l'algo
    cName=algo

    # Penalise
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, weights))
    clf= CatBoostClassifier(loss_function='MultiClassOneVsAll', class_weights=class_weights, verbose=False) # classification multiclasses

    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)

    # Overall Acuracy,  Kappa ; Precision; rappel, F1 score
    print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
    if NOM_SATELLITE == 'S2_SR':
        print("TYPE_SR", TYPE_SR)
    if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("Algo de fusion", ALGO_FUSION)
        if REPERTOIRE_SATELLITE == 'FUSION_SR':
            print("TYPE_SR", TYPE_SR)
    print(f"Précision globale {cName} : {accuracy_score(y_test,clf_pred)*100}")
    print(f"Indice de Kappa {cName} : {cohen_kappa_score(y_test,clf_pred)*100}")
    print(classification_report(y_test,clf_pred))


else:
    print("Mauvais choix d'algorithme")


In [ ]:
# CAS B)
##### OK LAST SANS ID_PARCELLE AVEC KFOLD
###### /////!!!!\\\\\\\\\ EXECUTER # TOUTES BANDES NON PARCELLE_ID
# TOUTES BANDES TEST SPLIT ak KFold + SHAP +/- ABLATION_STUDY
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score
#from sklearn.model_selection import GridSearchCV
#from sklearn.model_selection import GroupKFold # KFold Validation
from sklearn.model_selection import StratifiedKFold # KFold Validation with class imbalance


# K-Fold Validation
#kfold = GroupKFold(n_splits=5, shuffle=True, random_state=50)
#kfold = GroupKFold(n_splits=5)
skfold = StratifiedKFold(n_splits=5) # (n_splits-1)*(100/5) train et 1*(100/5) test
num_classes = len(np.unique(y))

# SHAP
import shap


# /!\ FAIT SEULEMENT POUR RF
algo = 'RF' # les valeurs sont 'SVM', 'RF' et 'CatBoost' ou implémenter le choix d'autres algorithmes au besoin

# TRAITEMENT GLOBAL
print('NO FIELD ID + STRATIFIED K-FOLD - TOUTES BANDES TEST SPLIT SANS VALIDATION')
print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
if NOM_SATELLITE == 'S2_SR':
    print("TYPE_SR", TYPE_SR)
if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
    print("Algo de fusion", ALGO_FUSION)
    if REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("TYPE_SR", TYPE_SR)
if ABLATION_STUDY == True:
    print('ABLATION STUDY')

if algo == 'SVM':
    # SVM
    from sklearn.svm import SVC

    # choix de l'algo
    cName=algo
    clf= SVC(kernel='rbf', class_weight='balanced', probability=True)

    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)

    # Overall Acuracy,  Kappa ; Precision; rappel, F1 score
    print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
    if NOM_SATELLITE == 'S2_SR':
        print("TYPE_SR", TYPE_SR)
    if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("Algo de fusion", ALGO_FUSION)
        if REPERTOIRE_SATELLITE == 'FUSION_SR':
            print("TYPE_SR", TYPE_SR)
    print(f"Précision globale {cName} : {accuracy_score(y_test,clf_pred)*100}")
    print(f"Indice de Kappa {cName} : {cohen_kappa_score(y_test,clf_pred)*100}")
    print(classification_report(y_test,clf_pred))

elif algo == 'RF':
    # Random Forest
    from sklearn.ensemble import RandomForestClassifier

    # choix de l'algo
    cName=algo


    # liste f1-score_values, X_test_values shap_values pour l'ensemble des 5 tests
    f1_scores_list = []
    X_test_values_list = []
    shap_values_list = []

    i = 1
    for train_indexes, test_indexes in skfold.split(x, y):
        # split
        X_train, X_test = x.iloc[train_indexes], x.iloc[test_indexes]
        y_train, y_test = y.iloc[train_indexes], y.iloc[test_indexes]

        # from scratch model to avoid data leakage
        # manual hyperparams
        #param_grid = {
        #        'n_estimators': [50, 100, 150],  # Number of trees
        #        'max_depth': [None, 10, 20],     # Maximum depth of trees
        #        'min_samples_split': [2, 5],     # Minimum samples required to split a node
        #        'min_samples_leaf': [1, 2],      # Minimum samples required to be at a leaf node
        #        'max_features': ['auto', 'sqrt', 'log2']  # Features to consider for best split
        #    }

        clf= RandomForestClassifier(
            class_weight='balanced',# balanced : penalization algorithm
            #n_estimators=100,
            #max_depth=20,
            #min_samples_split=5,
            #min_samples_leaf=1,
            #random_state=50, # reproductibilité, mais biais car modèle non iniatialisé à vide à chaque essai
        )
        # Apprentissage sur le jeu de données d'entraînement
        clf.fit(X_train,y_train)
        
        # Prédiction sur le jeu de données de test
        clf_pred= clf.predict(X_test)
        # score
        score = f1_score(y_test, clf_pred, average='weighted')
        f1_scores_list.append(score)
        print('Step', i, ': f1_score =', score * 100, '%')

        if ABLATION_STUDY == False: # pas de SHAP en cas d'ablation study
            # COMPUTE SHAP VALUES
            # create the SHAP explainer for RF model
            explainer = shap.TreeExplainer(clf)
           
            # calculate SHAP values for the test set
            shap_vals = explainer.shap_values(X_test)
    
            # NORMALISATION DU FORMAT SHAP (Gestion des versions SHAP array 3D vs list of 2D arrays)
            if isinstance(shap_vals, np.ndarray) and len(shap_vals.shape) == 3:
                # Si SHAP renvoie la forme (N_samples, N_features, N_classes)
                # On la transforme en liste de matrices 2D par classe : [ (N_samples, N_features), ... ]
                shap_vals = [shap_vals[:, :, c] for c in range(num_classes)]
            
            shap_values_list.append(shap_vals)
            X_test_values_list.append(X_test)
            
            # SHAP summary plot : feature importance + direction of impact (increase/decrease)
            #shap.summary_plot(shap_values[:,:, 1], X_test, feature_names = feature_names)

        i += 1
        

    # f1-score
    f1_scores_mean = np.mean(f1_scores_list) * 100
    f1_scores_std = np.std(f1_scores_list) * 100
    print('f1-score moyen :', f1_scores_mean, '% +/-', f1_scores_std)


    if ABLATION_STUDY == False: # pas de SHAP en cas d'ablation study
        # SHAP summary plot : feature importance + direction of impact (increase/decrease)
        X_test_global = pd.concat(X_test_values_list, axis=0).reset_index(drop = True)
        shap_values_global = []
        for classe_index in range(num_classes):
            class_shap = np.vstack([fold_shap[classe_index] for fold_shap in shap_values_list])
            shap_values_global.append(class_shap)
    
        print('SHAP VALUES SHAPE', shap_values_global[1].shape)
        print('X_test_global SHAPE', X_test_global.shape)
    
        # Affichage
        fig, axes = plt.subplots(2, 2, figsize=(6*num_classes, 7))
        axes = axes.flatten()
    
        for c in range(num_classes):
            plt.sca(axes[c])
            shap.summary_plot(
                shap_values_global[c],
                X_test_global,
                feature_names = feature_names,
                plot_size = None,
                show = False,
            )
            axes[c].set_title('Severity:' + str(c+1), fontsize=14, fontweight='bold')
    
        # SAVE FIG
        filename = 'NO_FIELD_ID-SKF-' + NOM_SATELLITE + '-' + TYPE_TRAITEMENT
        if NOM_SATELLITE == 'S2_SR':
            filename = filename + '-' + TYPE_SR
        if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
            filename = filename + '-' + ALGO_FUSION
            if REPERTOIRE_SATELLITE == 'FUSION_SR':
                filename = filename + '-' + TYPE_SR
        if ABLATION_STUDY == True:
            filename = filename + '-' + 'ablation_study'
        filename = filename + '.png'
        complete_filename = REPERTOIRE_IMAGES + '\\' + filename
        plt.savefig(complete_filename, bbox_inches='tight')
        plt.show()

    
elif algo == 'CatBoost':
    # XGBoost
    from catboost import CatBoostClassifier
    from sklearn.utils.class_weight import compute_class_weight

    # choix de l'algo
    cName=algo

    # Penalise
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, weights))
    clf= CatBoostClassifier(loss_function='MultiClassOneVsAll', class_weights=class_weights, verbose=False) # classification multiclasses

    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)

    # Overall Acuracy,  Kappa ; Precision; rappel, F1 score
    print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
    if NOM_SATELLITE == 'S2_SR':
        print("TYPE_SR", TYPE_SR)
    if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("Algo de fusion", ALGO_FUSION)
        if REPERTOIRE_SATELLITE == 'FUSION_SR':
            print("TYPE_SR", TYPE_SR)
    print(f"Précision globale {cName} : {accuracy_score(y_test,clf_pred)*100}")
    print(f"Indice de Kappa {cName} : {cohen_kappa_score(y_test,clf_pred)*100}")
    print(classification_report(y_test,clf_pred))


else:
    print("Mauvais choix d'algorithme")


In [ ]:
# CAS C)
##### OK LAST ak ID_PARCELLE + KFOLD
###### /////!!!!\\\\\\\\\ EXECUTER # TOUTES BANDES ak ID_PARCELLE
# TOUTES BANDES TEST SPLIT ak KFold + SHAP +/- ABLATION_STUDY
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import cohen_kappa_score
#from sklearn.model_selection import GridSearchCV
#from sklearn.model_selection import GroupKFold # KFold Validation
from sklearn.model_selection import StratifiedGroupKFold # KFold Validation with class imbalance


# K-Fold Validation
#kfold = GroupKFold(n_splits=5, shuffle=True, random_state=50)
#kfold = GroupKFold(n_splits=5)
sgkfold = StratifiedGroupKFold(n_splits=5) # (n_splits-1)*(100/5) train et 1*(100/5) test
num_classes = len(np.unique(y))

# SHAP
import shap


# /!\ FAIT SEULEMENT POUR RF
algo = 'RF' # les valeurs sont 'SVM', 'RF' et 'CatBoost' ou implémenter le choix d'autres algorithmes au besoin

# TRAITEMENT GLOBAL
print('FIELD ID + STRATIFIED GROUP K-FOLD - TOUTES BANDES TEST SPLIT SANS VALIDATION')
print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
if NOM_SATELLITE == 'S2_SR':
    print("TYPE_SR", TYPE_SR)
if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
    print("Algo de fusion", ALGO_FUSION)
    if REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("TYPE_SR", TYPE_SR)
if ABLATION_STUDY == True:
    print('ABLATION STUDY')

if algo == 'SVM':
    # SVM
    from sklearn.svm import SVC

    # choix de l'algo
    cName=algo
    clf= SVC(kernel='rbf', class_weight='balanced', probability=True)

    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)

    # Overall Acuracy,  Kappa ; Precision; rappel, F1 score
    print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
    if NOM_SATELLITE == 'S2_SR':
        print("TYPE_SR", TYPE_SR)
    if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("Algo de fusion", ALGO_FUSION)
        if REPERTOIRE_SATELLITE == 'FUSION_SR':
            print("TYPE_SR", TYPE_SR)
    print(f"Précision globale {cName} : {accuracy_score(y_test,clf_pred)*100}")
    print(f"Indice de Kappa {cName} : {cohen_kappa_score(y_test,clf_pred)*100}")
    print(classification_report(y_test,clf_pred))

elif algo == 'RF':
    # Random Forest
    from sklearn.ensemble import RandomForestClassifier

    # choix de l'algo
    cName=algo


    # liste f1-score_values, X_test_values shap_values pour l'ensemble des 5 tests
    f1_scores_list = []
    X_test_values_list = []
    shap_values_list = []

    i = 1
    for train_indexes, test_indexes in sgkfold.split(x, y, groups=groups):
        # split
        X_train, X_test = x.iloc[train_indexes], x.iloc[test_indexes]
        y_train, y_test = y.iloc[train_indexes], y.iloc[test_indexes]

        # from scratch model to avoid data leakage
        # manual hyperparams
        #param_grid = {
        #        'n_estimators': [50, 100, 150],  # Number of trees
        #        'max_depth': [None, 10, 20],     # Maximum depth of trees
        #        'min_samples_split': [2, 5],     # Minimum samples required to split a node
        #        'min_samples_leaf': [1, 2],      # Minimum samples required to be at a leaf node
        #        'max_features': ['auto', 'sqrt', 'log2']  # Features to consider for best split
        #    }

        clf= RandomForestClassifier(
            class_weight='balanced',# balanced : penalization algorithm
            #n_estimators=100,
            #max_depth=20,
            #min_samples_split=5,
            #min_samples_leaf=1,
            #random_state=50, # reproductibilité, mais biais car modèle non iniatialisé à vide à chaque essai
        )
        # Apprentissage sur le jeu de données d'entraînement
        clf.fit(X_train,y_train)
        
        # Prédiction sur le jeu de données de test
        clf_pred= clf.predict(X_test)
        # score
        score = f1_score(y_test, clf_pred, average='weighted')
        f1_scores_list.append(score)
        print('Step', i, ': f1_score =', score * 100, '%')
        
        if ABLATION_STUDY == False: # pas de SHAP en cas d'ablation study
            # COMPUTE SHAP VALUES
            # create the SHAP explainer for RF model
            explainer = shap.TreeExplainer(clf)
           
            # calculate SHAP values for the test set
            shap_vals = explainer.shap_values(X_test)
    
            # NORMALISATION DU FORMAT SHAP (Gestion des versions SHAP array 3D vs list of 2D arrays)
            if isinstance(shap_vals, np.ndarray) and len(shap_vals.shape) == 3:
                # Si SHAP renvoie la forme (N_samples, N_features, N_classes)
                # On la transforme en liste de matrices 2D par classe : [ (N_samples, N_features), ... ]
                shap_vals = [shap_vals[:, :, c] for c in range(num_classes)]
            
            shap_values_list.append(shap_vals)
            X_test_values_list.append(X_test)
            
            # SHAP summary plot : feature importance + direction of impact (increase/decrease)
            #shap.summary_plot(shap_values[:,:, 1], X_test, feature_names = feature_names)

        i += 1
        

    # f1-score
    f1_scores_mean = np.mean(f1_scores_list) * 100
    f1_scores_std = np.std(f1_scores_list) * 100
    print('f1-score moyen :', f1_scores_mean, '% +/-', f1_scores_std)


    if ABLATION_STUDY == False: # pas de SHAP en cas d'ablation study
        # SHAP summary plot : feature importance + direction of impact (increase/decrease)
        X_test_global = pd.concat(X_test_values_list, axis=0).reset_index(drop = True)
        shap_values_global = []
        for classe_index in range(num_classes):
            class_shap = np.vstack([fold_shap[classe_index] for fold_shap in shap_values_list])
            shap_values_global.append(class_shap)
    
        print('SHAP VALUES SHAPE', shap_values_global[1].shape)
        print('X_test_global SHAPE', X_test_global.shape)
    
        # Affichage
        fig, axes = plt.subplots(2, 2, figsize=(6*num_classes, 7))
        axes = axes.flatten()
    
        for c in range(num_classes):
            plt.sca(axes[c])
            shap.summary_plot(
                shap_values_global[c],
                X_test_global,
                feature_names = feature_names,
                plot_size = None,
                show = False,
            )
            axes[c].set_title('Severity:' + str(c+1), fontsize=14, fontweight='bold')
    
        # SAVE FIG
        filename = 'FIELD_ID-SGKF-' + NOM_SATELLITE + '-' + TYPE_TRAITEMENT
        if NOM_SATELLITE == 'S2_SR':
            filename = filename + '-' + TYPE_SR
        if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
            filename = filename + '-' + ALGO_FUSION
            if REPERTOIRE_SATELLITE == 'FUSION_SR':
                filename = filename + '-' + TYPE_SR
        if ABLATION_STUDY == True:
            filename = filename + '-' + 'ablation_study'
        filename = filename + '.png'
        complete_filename = REPERTOIRE_IMAGES + '\\' + filename
        plt.savefig(complete_filename, bbox_inches='tight')
        plt.show()

    
elif algo == 'CatBoost':
    # XGBoost
    from catboost import CatBoostClassifier
    from sklearn.utils.class_weight import compute_class_weight

    # choix de l'algo
    cName=algo

    # Penalise
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, weights))
    clf= CatBoostClassifier(loss_function='MultiClassOneVsAll', class_weights=class_weights, verbose=False) # classification multiclasses

    # Apprentissage sur le jeu de données d'entraînement
    clf.fit(X_train,y_train)
    # Prédiction sur le jeu de données de test
    clf_pred= clf.predict(X_test)

    # Overall Acuracy,  Kappa ; Precision; rappel, F1 score
    print('Satellite : {} - Traitement {}'.format(NOM_SATELLITE, TYPE_TRAITEMENT))
    if NOM_SATELLITE == 'S2_SR':
        print("TYPE_SR", TYPE_SR)
    if REPERTOIRE_SATELLITE == 'FUSION_SIMPLE' or REPERTOIRE_SATELLITE == 'FUSION_SR':
        print("Algo de fusion", ALGO_FUSION)
        if REPERTOIRE_SATELLITE == 'FUSION_SR':
            print("TYPE_SR", TYPE_SR)
    print(f"Précision globale {cName} : {accuracy_score(y_test,clf_pred)*100}")
    print(f"Indice de Kappa {cName} : {cohen_kappa_score(y_test,clf_pred)*100}")
    print(classification_report(y_test,clf_pred))


else:
    print("Mauvais choix d'algorithme")
